# DDPM for MNIST -- Training (from scratch, no diffusers)

Minimal Denoising Diffusion Probabilistic Model (Ho et al., 2020)
trained on MNIST, built entirely from this repo's own code -- no
`diffusers`, no external diffusion library. The pieces are split across
four version-controlled files (not duplicated in this notebook):

| piece            | file                              |
|------------------|------------------------------------|
| model (noise predictor) | `code/ddpm_mnist_model.py`    |
| noise scheduler  | `code/ddpm_mnist_scheduler.py`     |
| training loop    | `code/ddpm_mnist_train.py`         |
| sampling code    | `code/ddpm_mnist_sample.py` (used in the companion **evaluate** notebook, not here) |

Per this project's train/evaluate notebook split: **this notebook only
trains and checkpoints** (model + optimizer + epoch + configs, saved to
Google Drive). Sampling, the x_T -> x_0 trajectory trace, and the
reverse-step-count comparison all live in
`notebooks/colab/ddpm_mnist_evaluate.ipynb`, which loads the checkpoint
this notebook produces -- so new sampling diagnostics can be added and
rerun in seconds, with no retraining.

**Model** (`MnistUNet`, 2 downsampling stages, native 28x28 -- no
padding needed since 28 = 2x2x7):
```
input [B,1,28,28]
enc1 (1->c1) +time, relu -> [B,c1,28,28] = skip1 -> pool -> [B,c1,14,14]
enc2 (c1->c2) +time, relu -> [B,c2,14,14] = skip2 -> pool -> [B,c2, 7, 7]
bottleneck (c2->c3) +time, relu         -> [B,c3, 7, 7]
up2 + concat(skip2) -> dec2 (2c2->c2) +time, relu -> [B,c2,14,14]
up1 + concat(skip1) -> dec1 (2c1->c1) +time, relu -> [B,c1,28,28]
out_conv (c1->1)                         -> [B, 1,28,28]  = eps_hat
```
t enters via a fixed sinusoidal embedding -> learned MLP -> one linear
projection per stage, added as a per-channel bias (see
`code/ddpm_mnist_model.py` docstring for the full derivation).

**Training objective** (paper Eq. 14, "simple" loss):
`L_simple = E[ || eps - eps_theta(x_t, t) ||^2 ]`, with
`x_t = sqrt(alpha_bar_t) x0 + sqrt(1-alpha_bar_t) eps` (the scheduler's
`add_noise`), `t ~ Uniform{0,...,T-1}`, `eps ~ N(0,I)`.


## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

All model/scheduler/training code lives in `code/*.py`, not duplicated
here. Replace `REPO_URL` with this repository's URL if different.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Build model + scheduler, print shape trace and parameter count

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` shape at every point -- including every point t enters
the network -- before any real data is touched.

In [ ]:
from ddpm_mnist_model import MnistUNet, count_trainable_parameters
from ddpm_mnist_scheduler import DDPMScheduler

torch.manual_seed(0)

BASE_CHANNELS = 32
TIME_EMBEDDING_DIM = 64
NUM_TIMESTEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02

model = MnistUNet(base_channels=BASE_CHANNELS, time_embedding_dim=TIME_EMBEDDING_DIM).to(device)
scheduler = DDPMScheduler(NUM_TIMESTEPS, BETA_START, BETA_END, device=device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_x = torch.rand(4, 1, 28, 28, device=device)          # [B,1,28,28], synthetic, in [0,1]
sample_t = torch.randint(0, NUM_TIMESTEPS, (4,), device=device)  # [B]
_ = model(sample_x, sample_t, verbose=True)


## 4. Mount Google Drive, persistent dataset + checkpoint paths

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so both the MNIST download and the training checkpoint are
pointed at persistent Drive folders instead -- downloaded/saved once,
reused across sessions.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/thesis_data/mnist"
CHECKPOINT_PATH = "/content/drive/MyDrive/thesis_data/ddpm_mnist/checkpoint.pt"


## 5. Train

`train_one_epoch`/`evaluate` both implement the same denoising-MSE
objective; `evaluate` runs it on the held-out MNIST test split with no
gradient step, as an approximate monitoring signal (t and eps are
randomly resampled every call, so expect some run-to-run jitter at
fixed weights). A checkpoint (model + optimizer + epoch + model/scheduler
config + loss history) is saved every `CHECKPOINT_EVERY` epochs.

Set `RESUME = True` to continue from `CHECKPOINT_PATH` if it already
exists (e.g. after a Colab disconnect).

In [ ]:
from ddpm_mnist_train import get_mnist_dataloaders, train_one_epoch, evaluate, save_checkpoint, load_checkpoint

BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 20
CHECKPOINT_EVERY = 1
RESUME = False
SEED = 0

model_config = dict(in_channels=1, out_channels=1, base_channels=BASE_CHANNELS, time_embedding_dim=TIME_EMBEDDING_DIM)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

start_epoch = 0
train_loss_history, test_loss_history = [], []
if RESUME and os.path.exists(CHECKPOINT_PATH):
    checkpoint = load_checkpoint(CHECKPOINT_PATH, device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"]
    train_loss_history = checkpoint["train_loss_history"]
    test_loss_history = checkpoint["test_loss_history"]
    print(f"resumed from {CHECKPOINT_PATH} at epoch {start_epoch}")

train_loader, test_loader = get_mnist_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

for epoch in range(start_epoch, EPOCHS):
    train_mse = train_one_epoch(model, scheduler, train_loader, optimizer, device)
    test_mse = evaluate(model, scheduler, test_loader, device)
    train_loss_history.append(train_mse)
    test_loss_history.append(test_mse)
    print(f"epoch {epoch + 1:3d}/{EPOCHS}  train_mse={train_mse:.4f}  test_mse={test_mse:.4f}")

    if (epoch + 1) % CHECKPOINT_EVERY == 0 or (epoch + 1) == EPOCHS:
        save_checkpoint(
            CHECKPOINT_PATH, model, optimizer, epoch=epoch + 1,
            model_config=model_config, scheduler_config=scheduler.config(),
            train_loss_history=train_loss_history, test_loss_history=test_loss_history,
            seed=SEED,
        )


## 6. Denoising-MSE loss curves (quantitative result over training)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(train_loss_history, label="train")
ax.plot(test_loss_history, label="test")
ax.set_xlabel("epoch")
ax.set_ylabel("denoising MSE (L_simple estimate)")
ax.set_title("DDPM training loss")
ax.legend()
fig.tight_layout()
plt.show()

print(f"final test_mse = {test_loss_history[-1]:.4f}")


## 7. Next step

Open `notebooks/colab/ddpm_mnist_evaluate.ipynb`, point it at the same
`CHECKPOINT_PATH`, and run sampling / the x_T -> x_0 trajectory trace /
the reverse-step-count comparison -- no retraining required.